# Algoritmos de cadenas · Autómatas, transformadas y alineamiento

Tres familias de algoritmos de texto. El **autómata de Aho-Corasick** busca muchos patrones a la vez en una sola pasada, y el **autómata de sufijos** reconoce todas las subcadenas de un texto con a lo sumo $2n - 1$ estados. La **transformada de Burrows-Wheeler** reordena un texto de forma reversible para que las letras iguales queden juntas, lo que favorece la compresión RLE. Por último, la **distancia de Hamming** y el **alineamiento global de Needleman-Wunsch** ($O(nm)$) miden cuánto se parecen dos secuencias. Cada algoritmo se contrasta con una solución directa. Ideas de [cp-algorithms](https://cp-algorithms.com/string/aho_corasick.html) y del [curso de bioinformática de Rosalind](https://rosalind.info/problems/list-view/).

In [1]:
import random
from collections import deque
from itertools import groupby

## 1. Construir el autómata de Aho-Corasick

Construir el autómata para un conjunto de patrones: un trie con los patrones, un enlace de fallo por nodo (el nodo del sufijo propio más largo de su prefijo que también está en el trie), calculado en anchura, y la lista de patrones que terminan en cada nodo o en sus fallos. Mostrar la tabla de nodos y comprobar cada enlace de fallo probando todos los sufijos.

In [2]:
class AhoCorasick:
    def __init__(self, patrones):
        self.patrones = list(patrones)
        self.hijos, self.fallo, self.salida, self.prefijo = [{}], [0], [[]], [""]
        for k, patron in enumerate(self.patrones):
            nodo = 0
            for c in patron:
                if c not in self.hijos[nodo]:
                    self.hijos[nodo][c] = len(self.hijos)
                    self.hijos.append({})
                    self.fallo.append(0)
                    self.salida.append([])
                    self.prefijo.append(self.prefijo[nodo] + c)
                nodo = self.hijos[nodo][c]
            self.salida[nodo].append(k)
        cola = deque(self.hijos[0].values())
        while cola:
            u = cola.popleft()
            for c, v in self.hijos[u].items():
                f = self.fallo[u]
                while f and c not in self.hijos[f]:
                    f = self.fallo[f]
                self.fallo[v] = self.hijos[f].get(c, 0) if u else 0
                self.salida[v] = self.salida[v] + self.salida[self.fallo[v]]
                cola.append(v)

    def buscar(self, texto):
        """Devuelve (posición, patrón) de cada aparición y el número de pasos del autómata."""
        estado, encontrados, pasos = 0, [], 0
        for i, c in enumerate(texto):
            while estado and c not in self.hijos[estado]:
                estado = self.fallo[estado]
                pasos += 1
            estado = self.hijos[estado].get(c, 0)
            pasos += 1
            for k in self.salida[estado]:
                p = self.patrones[k]
                encontrados.append((i - len(p) + 1, p))
        return sorted(encontrados), pasos


patrones = ["mar", "amar", "ama", "rama", "marea", "are"]
ac = AhoCorasick(patrones)
print(f"{'nodo':>4}  {'prefijo':8} {'fallo':8} salidas")
for v in range(len(ac.hijos)):
    print(f"{v:4}  {ac.prefijo[v]:8} {ac.prefijo[ac.fallo[v]]!r:8} {[patrones[k] for k in ac.salida[v]]}")

prefijos = set(ac.prefijo)
for v, p in enumerate(ac.prefijo):
    esperado = next((p[i:] for i in range(1, len(p) + 1) if p[i:] in prefijos), "")
    assert ac.prefijo[ac.fallo[v]] == esperado

nodo  prefijo  fallo    salidas
   0           ''       []
   1  m        ''       []
   2  ma       'a'      []
   3  mar      'ar'     ['mar']
   4  a        ''       []
   5  am       'm'      []
   6  ama      'ma'     ['ama']
   7  amar     'mar'    ['amar', 'mar']
   8  r        ''       []
   9  ra       'a'      []
  10  ram      'am'     []
  11  rama     'ama'    ['rama', 'ama']
  12  mare     'are'    ['are']
  13  marea    'a'      ['marea']
  14  ar       'r'      []
  15  are      ''       ['are']


## 2. Buscar todos los patrones en una pasada

Buscar los patrones anteriores en una frase, incluidas las apariciones solapadas, y comparar con la fuerza bruta (`startswith` de cada patrón en cada posición).

In [3]:
def fuerza_bruta(texto, patrones):
    return sorted((i, p) for p in patrones for i in range(len(texto)) if texto.startswith(p, i))


frase = "la ramamareaba al amarrar la marea en la rama del mar de amarillo"
encontrados, _ = ac.buscar(frase)
for i, p in encontrados:
    print(f"{i:3} {p}")
assert encontrados == fuerza_bruta(frase, patrones)

  3 rama
  4 ama
  6 ama
  6 amar
  7 mar
  7 marea
  8 are
 18 ama
 18 amar
 19 mar
 29 mar
 29 marea
 30 are
 41 rama
 42 ama
 50 mar
 57 ama
 57 amar
 58 mar


## 3. Muchos patrones: autómata frente a búsquedas sueltas

Generar una secuencia de ADN de 20 000 bases y 200 patrones de entre 6 y 12 bases (la mitad tomados del propio texto). Contar los pasos del autómata (transiciones y saltos de fallo) y las comparaciones de caracteres de buscar cada patrón por separado con el método ingenuo. Comprobar que dan las mismas apariciones y que el autómata hace menos de $2n$ pasos sea cual sea el número de patrones.

In [4]:
def busquedas_sueltas(texto, patrones):
    encontrados, comparaciones = [], 0
    for p in patrones:
        for i in range(len(texto) - len(p) + 1):
            j = 0
            while j < len(p):
                comparaciones += 1
                if texto[i + j] != p[j]:
                    break
                j += 1
            if j == len(p):
                encontrados.append((i, p))
    return sorted(encontrados), comparaciones


rng = random.Random(42)
adn = "".join(rng.choices("acgt", k=20_000))
muchos = set()
while len(muchos) < 200:
    largo = rng.randint(6, 12)
    if len(muchos) % 2:
        i = rng.randrange(len(adn) - largo)
        muchos.add(adn[i:i + largo])
    else:
        muchos.add("".join(rng.choices("acgt", k=largo)))
muchos = sorted(muchos)

r_ac, pasos = AhoCorasick(muchos).buscar(adn)
r_sueltas, comparaciones = busquedas_sueltas(adn, muchos)
print(f"Apariciones: {len(r_ac)}")
print(f"Pasos de Aho-Corasick:            {pasos:>10,}")
print(f"Comparaciones búsquedas sueltas:  {comparaciones:>10,}  ({comparaciones / pasos:.0f} veces más)")
assert r_ac == r_sueltas and pasos < 2 * len(adn)

Apariciones: 283
Pasos de Aho-Corasick:                36,744
Comparaciones búsquedas sueltas:   5,332,255  (145 veces más)


## 4. Censurar palabras

Usar el autómata para tapar con asteriscos todas las letras que forman parte de alguna palabra prohibida de una lista, aunque las apariciones se solapen o estén dentro de otras palabras. Comprobar el resultado con una máscara calculada por fuerza bruta.

In [5]:
def censurar(texto, prohibidas):
    marcas = [0] * (len(texto) + 1)
    for i, p in AhoCorasick(prohibidas).buscar(texto.lower())[0]:
        marcas[i] += 1
        marcas[i + len(p)] -= 1
    salida, abiertas = [], 0
    for c, m in zip(texto, marcas):
        abiertas += m
        salida.append("*" if abiertas else c)
    return "".join(salida)


prohibidas = ["chacho", "mago", "tolete", "magua"]
mensaje = "Chacho, el mago de la magua dijo que el TOLETE era otro; chachomago!"
censurado = censurar(mensaje, prohibidas)
print(mensaje)
print(censurado)
tapadas = {i + k for i, p in fuerza_bruta(mensaje.lower(), prohibidas) for k in range(len(p))}
assert censurado == "".join("*" if i in tapadas else c for i, c in enumerate(mensaje))

Chacho, el mago de la magua dijo que el TOLETE era otro; chachomago!
******, el **** de la ***** dijo que el ****** era otro; **********!


## 5. Autómata de sufijos

Construir el autómata de sufijos de una cadena de forma incremental (con estados clonados cuando hace falta) y comprobar con cadenas aleatorias que reconoce exactamente las subcadenas, que tiene como mucho $2n - 1$ estados y $3n - 4$ transiciones (para $n \ge 3$).

In [6]:
def automata_sufijos(s):
    sig, enlace, largo, ultimo = [{}], [-1], [0], 0
    for c in s:
        actual = len(largo)
        sig.append({})
        enlace.append(0)
        largo.append(largo[ultimo] + 1)
        p = ultimo
        while p != -1 and c not in sig[p]:
            sig[p][c] = actual
            p = enlace[p]
        if p != -1:
            q = sig[p][c]
            if largo[p] + 1 == largo[q]:
                enlace[actual] = q
            else:
                clon = len(largo)
                sig.append(dict(sig[q]))
                enlace.append(enlace[q])
                largo.append(largo[p] + 1)
                while p != -1 and sig[p].get(c) == q:
                    sig[p][c] = clon
                    p = enlace[p]
                enlace[q] = enlace[actual] = clon
        ultimo = actual
    return sig, enlace, largo


def reconoce(automata, t):
    sig, estado = automata[0], 0
    for c in t:
        if c not in sig[estado]:
            return False
        estado = sig[estado][c]
    return True


s = "tararear"
automata = automata_sufijos(s)
sig = automata[0]
print(f"{s!r}: {len(sig)} estados, {sum(map(len, sig))} transiciones")
for t in ["rare", "tara", "rea", "art", "", "tararear", "tarareas"]:
    print(f"  {t!r:11} {reconoce(automata, t)}")

rng = random.Random(8)
for _ in range(200):
    s = "".join(rng.choices(rng.choice(["ab", "abc"]), k=rng.randint(3, 30)))
    automata = automata_sufijos(s)
    n, sig = len(s), automata[0]
    assert len(sig) <= 2 * n - 1 and sum(map(len, sig)) <= 3 * n - 4
    for _ in range(20):
        t = "".join(rng.choices("abc", k=rng.randint(0, 6)))
        assert reconoce(automata, t) == (t in s)

'tararear': 11 estados, 14 transiciones
  'rare'      True
  'tara'      True
  'rea'       True
  'art'       False
  ''          True
  'tararear'  True
  'tarareas'  False


## 6. Subcadenas distintas con el autómata

Cada estado del autómata representa las subcadenas cuyas longitudes van de `largo[enlace] + 1` a `largo`, y cada subcadena está en un único estado. Contar las subcadenas distintas como $\sum (\text{largo}_v - \text{largo}_{\text{enlace}(v)})$ y comprobarlo con un conjunto.

In [7]:
def distintas_automata(s):
    _, enlace, largo = automata_sufijos(s)
    return sum(largo[v] - largo[enlace[v]] for v in range(1, len(largo)))


for s in ["tararear", "aaaa", "papagayo"]:
    print(f"{s:10} {distintas_automata(s)}")
rng = random.Random(9)
for _ in range(200):
    s = "".join(rng.choices("abc", k=rng.randint(1, 40)))
    assert distintas_automata(s) == len({s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)})

tararear   30
aaaa       4
papagayo   32


## 7. Subcadena común más larga con el autómata

Construir el autómata de la primera cadena y recorrer la segunda: si una letra no tiene transición, retroceder por los enlaces de sufijo (acortando la coincidencia) hasta que la tenga. La coincidencia más larga vista es la subcadena común más larga, en $O(n + m)$. Comparar con la programación dinámica.

In [8]:
def comun_automata(a, b):
    sig, enlace, largo = automata_sufijos(a)
    v = l = mejor = fin = 0
    for i, c in enumerate(b):
        while v and c not in sig[v]:
            v = enlace[v]
            l = largo[v]
        if c in sig[v]:
            v, l = sig[v][c], l + 1
        if l > mejor:
            mejor, fin = l, i + 1
    return b[fin - mejor:fin]


def comun_dp(a, b):
    mejor, fila = 0, [0] * (len(b) + 1)
    for x in a:
        nueva = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            if x == y:
                nueva[j] = fila[j - 1] + 1
                mejor = max(mejor, nueva[j])
        fila = nueva
    return mejor


print(repr(comun_automata("timple canario", "el canario y el timple")))
rng = random.Random(10)
for _ in range(300):
    a = "".join(rng.choices("acgt", k=rng.randint(1, 40)))
    b = "".join(rng.choices("acgt", k=rng.randint(1, 40)))
    c = comun_automata(a, b)
    assert len(c) == comun_dp(a, b) and c in a and c in b

' canario'


## 8. Transformada de Burrows-Wheeler por rotaciones

Añadir al texto un centinela menor que cualquier letra (el carácter `"\x00"`, que se muestra como `$`), ordenar todas sus rotaciones y quedarse con la última columna. Mostrar la matriz de rotaciones de `"tararear"`.

In [9]:
CENTINELA = "\x00"  # "$" no vale: el espacio y otros signos son menores


def ver(s):
    return s.replace(CENTINELA, "$")


def bwt_rotaciones(texto):
    s = texto + CENTINELA
    rotaciones = sorted(s[i:] + s[:i] for i in range(len(s)))
    return "".join(r[-1] for r in rotaciones), rotaciones


bwt, rotaciones = bwt_rotaciones("tararear")
for r in rotaciones:
    print(ver(f"{r[0]} {r[1:-1]} {r[-1]}"))
print("BWT:", ver(bwt))
assert sorted(bwt) == sorted("tararear" + CENTINELA)

$ tararea r
a r$tarar e
a rarear$ t
a rear$ta r
e ar$tara r
r $tarare a
r arear$t a
r ear$tar a
t ararear $
BWT: retrraaa$


## 9. BWT desde el array de sufijos

Ordenar las rotaciones del texto con el centinela equivale a ordenar sus sufijos, porque el centinela es único y el menor. Calcular la transformada como `s[sa[i] - 1]` a partir del array de sufijos, sin construir las rotaciones, y comprobarlo con el método anterior.

In [10]:
def bwt_sufijos(texto):
    s = texto + CENTINELA
    sa = sorted(range(len(s)), key=lambda i: s[i:])
    return "".join(s[i - 1] for i in sa)


print(ver(bwt_sufijos("tararear")))
rng = random.Random(12)
for _ in range(300):
    t = "".join(rng.choices("abcn", k=rng.randint(0, 40)))
    assert bwt_sufijos(t) == bwt_rotaciones(t)[0]

retrraaa$


## 10. Inversa de la BWT

Recuperar el texto original a partir de la última columna con la correspondencia LF: la $k$-ésima aparición de una letra en la última columna es la $k$-ésima aparición de esa letra en la primera (que es la última ordenada). Partiendo de la fila del centinela, ir hacia atrás letra a letra. Comprobar la ida y vuelta en textos aleatorios.

In [11]:
def bwt_inversa(ultima):
    primera_pos, vistas, rango = {}, {}, []
    for i, c in enumerate(sorted(ultima)):
        primera_pos.setdefault(c, i)
    for c in ultima:
        rango.append(vistas.get(c, 0))
        vistas[c] = rango[-1] + 1
    fila, letras = 0, []  # la fila 0 empieza por el centinela
    for _ in range(len(ultima) - 1):
        c = ultima[fila]
        letras.append(c)
        fila = primera_pos[c] + rango[fila]
    return "".join(reversed(letras))


print(bwt_inversa(bwt))
rng = random.Random(14)
for _ in range(300):
    t = "".join(rng.choices("abcn ", k=rng.randint(0, 60)))
    assert bwt_inversa(bwt_sufijos(t)) == t

tararear


## 11. BWT y compresión RLE

Codificar un texto repetitivo con RLE (pares letra y longitud de racha) antes y después de aplicarle la BWT, comparar el número de rachas y comprobar que deshacer RLE y la BWT devuelve el texto original.

In [12]:
def rle(texto):
    return [(c, len(list(g))) for c, g in groupby(texto)]


def rle_inversa(pares):
    return "".join(c * n for c, n in pares)


copla = ("papa arrugada con mojo picón, papa sancochada con mojo verde, "
         "papa bonita con mojo rojo y papa negra con mojo de cilantro. ") * 3
directo = rle(copla)
transformado = rle(bwt_sufijos(copla))
print(f"Letras: {len(copla)}; rachas sin BWT: {len(directo)}; rachas con BWT: {len(transformado)}")
print("Primeras rachas con BWT:", [(ver(c), n) for c, n in transformado[:12]])
assert bwt_inversa(rle_inversa(transformado)) == copla
assert len(transformado) < len(directo) / 2

Letras: 369; rachas sin BWT: 366; rachas con BWT: 79
Primeras rachas con BWT: [(' ', 1), ('.', 1), ('a', 6), ('e', 3), ('a', 12), ('o', 3), ('n', 12), ('a', 3), ('.', 2), (',', 3), ('y', 3), (',', 3)]


## 12. Distancia de Hamming

Escribir la distancia de Hamming entre dos cadenas de la misma longitud (número de posiciones en que difieren), que lance `ValueError` si las longitudes no coinciden. Usarla para encontrar en una lista de lecturas de ADN las que están a distancia 1 o menos de una consulta y comprobarlo comparando posición a posición.

In [13]:
def hamming(a, b):
    if len(a) != len(b):
        raise ValueError(f"longitudes distintas: {len(a)} y {len(b)}")
    return sum(x != y for x, y in zip(a, b))


print(hamming("guagua", "guagüa"), hamming("1011101", "1001001"))
try:
    hamming("gofio", "gofios")
except ValueError as e:
    print("Error:", e)

rng = random.Random(15)
consulta = "acgtacgtac"
lecturas = ["".join(c if rng.random() > 0.1 else rng.choice("acgt") for c in consulta) for _ in range(12)]
for l in lecturas:
    d = hamming(consulta, l)
    print(f"{l}  {d}" + ("  <-" if d <= 1 else ""))
    assert d == len([i for i in range(len(l)) if l[i] != consulta[i]])

1 2
Error: longitudes distintas: 5 y 6
aagtccgtac  2
acgtacgtac  0  <-
acgtacgtaa  1  <-
acgtatgtac  1  <-
gcgtacgtac  1  <-
acggacgtac  1  <-
acatacgtac  1  <-
tcgtacgtag  2
acgtacgtag  1  <-
acgtacgtac  0  <-
ccgtacgtac  1  <-
acgtacgtac  0  <-


## 13. Needleman-Wunsch: tabla de puntuaciones

Calcular la puntuación del mejor alineamiento global de dos secuencias con coincidencia $+1$, sustitución $-1$ y hueco $-2$: `F[i][j]` es el máximo entre alinear las dos letras, poner un hueco en la segunda o ponerlo en la primera. Mostrar la tabla para `"GATACA"` y `"GTTACCA"` y comparar la puntuación con la recursión que prueba todos los alineamientos.

In [14]:
COINCIDE, SUSTITUYE, HUECO = 1, -1, -2


def puntos(x, y):
    return COINCIDE if x == y else SUSTITUYE


def tabla_nw(a, b):
    F = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        F[i][0] = i * HUECO
    for j in range(1, len(b) + 1):
        F[0][j] = j * HUECO
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            F[i][j] = max(F[i - 1][j - 1] + puntos(a[i - 1], b[j - 1]), F[i - 1][j] + HUECO, F[i][j - 1] + HUECO)
    return F


def todos_los_alineamientos(a, b):
    if not a or not b:
        return (len(a) + len(b)) * HUECO
    return max(todos_los_alineamientos(a[:-1], b[:-1]) + puntos(a[-1], b[-1]),
               todos_los_alineamientos(a[:-1], b) + HUECO, todos_los_alineamientos(a, b[:-1]) + HUECO)


a, b = "GATACA", "GTTACCA"
F = tabla_nw(a, b)
print("      " + "".join(f"{c:>4}" for c in "-" + b))
for i, fila in enumerate(F):
    print(f"{('-' + a)[i]:>4}  " + "".join(f"{v:4}" for v in fila))
print("Puntuación:", F[-1][-1])
assert F[-1][-1] == todos_los_alineamientos(a, b)
rng = random.Random(16)
for _ in range(30):
    x = "".join(rng.choices("ACGT", k=rng.randint(0, 6)))
    y = "".join(rng.choices("ACGT", k=rng.randint(0, 6)))
    assert tabla_nw(x, y)[-1][-1] == todos_los_alineamientos(x, y)

         -   G   T   T   A   C   C   A
   -     0  -2  -4  -6  -8 -10 -12 -14
   G    -2   1  -1  -3  -5  -7  -9 -11
   A    -4  -1   0  -2  -2  -4  -6  -8
   T    -6  -3   0   1  -1  -3  -5  -7
   A    -8  -5  -2  -1   2   0  -2  -4
   C   -10  -7  -4  -3   0   3   1  -1
   A   -12  -9  -6  -5  -2   1   2   2
Puntuación: 2


## 14. Reconstruir el alineamiento

Recorrer la tabla desde la esquina final hacia atrás, eligiendo en cada casilla el movimiento que explica su valor, para obtener las dos secuencias alineadas con guiones en los huecos. Mostrarlas con una línea intermedia (`|` coincidencia, `.` sustitución) y comprobar que, al quitar los guiones, salen las secuencias originales y que la puntuación del alineamiento coincide con la de la tabla.

In [15]:
def alinear(a, b):
    F = tabla_nw(a, b)
    i, j, sup, inf = len(a), len(b), [], []
    while i or j:
        if i and j and F[i][j] == F[i - 1][j - 1] + puntos(a[i - 1], b[j - 1]):
            sup.append(a[i - 1]); inf.append(b[j - 1]); i -= 1; j -= 1
        elif i and F[i][j] == F[i - 1][j] + HUECO:
            sup.append(a[i - 1]); inf.append("-"); i -= 1
        else:
            sup.append("-"); inf.append(b[j - 1]); j -= 1
    return "".join(reversed(sup)), "".join(reversed(inf)), F[-1][-1]


def puntuar(sup, inf):
    return sum(HUECO if "-" in (x, y) else puntos(x, y) for x, y in zip(sup, inf))


def mostrar(sup, inf):
    medio = "".join(" " if "-" in (x, y) else "|" if x == y else "." for x, y in zip(sup, inf))
    print(sup, medio, inf, sep="\n")


for a, b in [("GATACA", "GTTACCA"), ("TENERIFE", "TERERIFE"), ("FUERTEVENTURA", "FUERTEAVENTURA")]:
    sup, inf, total = alinear(a, b)
    mostrar(sup, inf)
    print(f"Puntuación: {total}\n")
    assert sup.replace("-", "") == a and inf.replace("-", "") == b
    assert len(sup) == len(inf) and puntuar(sup, inf) == total

GATA-CA
|.|| ||
GTTACCA
Puntuación: 2

TENERIFE
||.|||||
TERERIFE
Puntuación: 6

FUERTE-VENTURA
|||||| |||||||
FUERTEAVENTURA
Puntuación: 11



## 15. Hamming frente a alineamiento

Comparar las dos medidas en una secuencia y la misma desplazada una posición: la distancia de Hamming es grande porque compara posición a posición, mientras que el alineamiento descubre que solo sobran una letra al principio y otra al final. Comprobar que la puntuación del alineamiento supera a la de alinear sin huecos, que es $n - 2\,d_H$.

In [16]:
original = "ACGTTGCAACGTTGCA"
desplazada = original[1:] + "G"
d = hamming(original, desplazada)
sup, inf, total = alinear(original, desplazada)
print(f"Hamming: {d} de {len(original)}")
mostrar(sup, inf)
print(f"Puntuación con huecos: {total}; sin huecos: {len(original) - 2 * d}")
assert puntuar(original, desplazada) == len(original) - 2 * d
assert total > len(original) - 2 * d

Hamming: 13 de 16
ACGTTGCAACGTTGCA-
 ||||||||||||||| 
-CGTTGCAACGTTGCAG
Puntuación con huecos: 11; sin huecos: -10
